# Vanilla RNN — One‑Stop Math + Deep Theory + End‑to‑End Worked Example (PyTorch)

This notebook is meant to be your **one-stop recap** for a *plain* (vanilla) RNN:
- **Exact math** (paper notation)
- **Detailed explanation of every equation and symbol**
- **Shape/dimension sanity checks**
- **Why vanishing/exploding gradients happens**
- **End-to-end worked numeric example in PyTorch**
- **Autograd gradients + optional finite-difference check**

> Tip for note-taking: For each section, copy the boxed equations and then write the *plain-English meaning* next to them.

## 0) Core idea: what an RNN does

An RNN processes a sequence one element at a time:

- At time step \(t\), it reads the current input \(x_t\).
- It also carries a **memory** from the previous step \(h_{t-1}\) (the hidden state).
- It combines them to produce the new hidden state \(h_t\).

This creates a computation chain:

$$
x_1 \rightarrow h_1 \rightarrow h_2 \rightarrow \dots \rightarrow h_T
$$

The crucial property:

**The same parameters are reused at every time step**.  
That’s why it’s called *recurrent*.

## 1) Notation and shapes (how to read paper symbolism)

### 1.1 What does “\(\in\)” mean?  
\(\in\) means “is an element of / belongs to”. Example:
$$
W_{hh} \in \mathbb{R}^{d_h \times d_h}
$$
reads as: “\(W_{hh}\) is a real-valued matrix of shape \(d_h \times d_h\).”

### 1.2 Common symbols you’ll see
- \(\mathbb{R}\): real numbers
- \(\mathbb{R}^{m \times n}\): set of real matrices with \(m\) rows, \(n\) columns
- \(^\top\): transpose
- \(\odot\): elementwise (Hadamard) product
- \(\sigma(\cdot)\): sigmoid (used later in LSTM/GRU)
- \(\tanh(\cdot)\): hyperbolic tangent
- \(\prod\): product over a range (multiplication, like \(\sum\) is addition)

### 1.3 Shapes used in this notebook
We assume:
- Input vector: \(x_t \in \mathbb{R}^{d_x}\)
- Hidden state: \(h_t \in \mathbb{R}^{d_h}\)
- Logits (pre-softmax): \(o_t \in \mathbb{R}^{d_y}\)

In code, we often use **column vectors**: \((d \times 1)\).  
So think \(x_t\) as \((d_x \times 1)\), \(h_t\) as \((d_h \times 1)\).

Parameters:
$$
W_{xh} \in \mathbb{R}^{d_h \times d_x},\quad
W_{hh} \in \mathbb{R}^{d_h \times d_h},\quad
b_h \in \mathbb{R}^{d_h}
$$
$$
W_{hy} \in \mathbb{R}^{d_y \times d_h},\quad
b_y \in \mathbb{R}^{d_y}
$$

**Meaning of subscripts**:
- \(W_{xh}\): input \(x\) → hidden \(h\)
- \(W_{hh}\): hidden \(h\) → hidden \(h\) (the recurrent matrix)
- \(W_{hy}\): hidden \(h\) → output \(y\)

## 2) Forward pass (the vanilla RNN equations + explanation)

### 2.1 Pre-activation (linear part)
$$
a_t = W_{xh}x_t + W_{hh}h_{t-1} + b_h
$$

**What it means**
- \(W_{xh}x_t\): converts the current input into hidden-space features.
- \(W_{hh}h_{t-1}\): brings forward “memory” from the previous hidden state.
- \(b_h\): shifts the activation (bias).

**Shape check** (column-vector view):
- \(W_{xh}(d_h\times d_x)\cdot x_t(d_x\times 1) = (d_h\times 1)\)
- \(W_{hh}(d_h\times d_h)\cdot h_{t-1}(d_h\times 1) = (d_h\times 1)\)
So \(a_t\) is \((d_h\times 1)\), consistent.

### 2.2 Nonlinearity (state update)
$$
h_t = \tanh(a_t)
$$

**What it means**
- \(\tanh\) keeps values bounded in \((-1,1)\), which helps stability.
- The hidden state \(h_t\) is the RNN’s “memory representation” at time \(t\).

### 2.3 Output (logits) and probabilities
$$
o_t = W_{hy}h_t + b_y
$$
If doing classification at each step, we convert logits to probabilities via softmax:
$$
\hat y_t = \mathrm{softmax}(o_t)
$$

**Why logits?**  
Logits are raw scores. Softmax turns them into probabilities that sum to 1.

## 3) Loss (cross-entropy) and the key gradient identity

For many-to-many classification, each time step has a target \(y_t\) (often one-hot).

Per-step cross-entropy:
$$
\ell_t = -y_t^\top \log(\hat y_t)
$$
Total loss:
$$
L = \sum_{t=1}^T \ell_t
$$

### Key identity (softmax + cross-entropy)
A very common paper result:
$$
\frac{\partial \ell_t}{\partial o_t} = \hat y_t - y_t
$$

**Interpretation**
- If the model assigns too much probability to the wrong class, the gradient pushes logits down for that class.
- The gradient is “prediction minus truth”.

## 4) Backpropagation Through Time (BPTT): detailed meaning

Because \(h_t\) affects:
1) the loss at time \(t\), **and**
2) all future hidden states \(h_{t+1}, h_{t+2}, \dots\),

the gradient wrt \(h_t\) has **two sources**.

Define two common “delta” terms (papers use these all the time):

- Output delta:
$$
\delta_t^o \triangleq \frac{\partial \ell_t}{\partial o_t}
$$

- Hidden delta:
$$
\delta_t^h \triangleq \frac{\partial L}{\partial h_t}
$$

- Pre-activation delta:
$$
\delta_t^a \triangleq \frac{\partial L}{\partial a_t}
$$

### 4.1 Output delta
Using the identity above:
$$
\delta_t^o = \hat y_t - y_t
$$

### 4.2 Hidden delta recursion (THIS is the “through time” part)
$$
\delta_t^h = W_{hy}^\top \delta_t^o + W_{hh}^\top \delta_{t+1}^a
$$

Read this as:
- **First term**: gradient flowing into \(h_t\) from the output at time \(t\).
- **Second term**: gradient flowing into \(h_t\) from the future, because \(h_t\) influences \(h_{t+1}\).

At the final step \(t=T\), there is no future term, so \(\delta_{T+1}^a = 0\).

### 4.3 Backprop through tanh
Because \(h_t = \tanh(a_t)\), elementwise:
$$
\frac{\partial h_t}{\partial a_t} = 1 - h_t \odot h_t
$$
So:
$$
\delta_t^a = \delta_t^h \odot (1 - h_t \odot h_t)
$$

### 4.4 Parameter gradients
From \(o_t = W_{hy}h_t + b_y\):
$$
\frac{\partial L}{\partial W_{hy}} = \sum_{t=1}^T \delta_t^o h_t^\top,\quad
\frac{\partial L}{\partial b_y} = \sum_{t=1}^T \delta_t^o
$$

From \(a_t = W_{xh}x_t + W_{hh}h_{t-1} + b_h\):
$$
\frac{\partial L}{\partial W_{xh}} = \sum_{t=1}^T \delta_t^a x_t^\top
$$
$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \delta_t^a h_{t-1}^\top
$$
$$
\frac{\partial L}{\partial b_h} = \sum_{t=1}^T \delta_t^a
$$

**Paper-reading skill**: whenever you see terms like \(\delta_t^a x_t^\top\), it’s almost always “error signal × input”.

## 5) Vanishing and exploding gradients (why RNNs struggle on long sequences)

The key chain-rule expression:
$$
\frac{\partial L}{\partial h_{t-k}}
=
\frac{\partial L}{\partial h_t}
\prod_{i=t-k+1}^{t}
\frac{\partial h_i}{\partial h_{i-1}}
$$

- The product \(\prod\) means you multiply Jacobians from step to step.
- If those Jacobians have singular values < 1, repeated multiplication shrinks gradients (vanishing).
- If > 1, repeated multiplication grows gradients (exploding).

For vanilla RNN:
$$
\frac{\partial h_i}{\partial h_{i-1}}
=
\mathrm{diag}(1-h_i^2)\,W_{hh}
$$

So the recurrent matrix \(W_{hh}\) sits inside the repeated product — that’s why its spectral properties matter so much.

## 6) End-to-end worked example in PyTorch (with gradients)

In the code below we:
- build the RNN computation explicitly (no `nn.RNN` yet)
- compute cross-entropy at each step
- call `loss.backward()`
- inspect `param.grad`

This matches the math directly.

In [1]:
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)

device = "cpu"

# Tiny dimensions
d_x, d_h, d_y = 2, 3, 2
T = 3

# Parameters (small numbers for readability)
W_xh = torch.tensor([[ 0.10, -0.20],
                     [ 0.00,  0.30],
                     [-0.10,  0.20]], device=device, requires_grad=True)   # (d_h, d_x)

W_hh = torch.tensor([[ 0.20,  0.00, -0.10],
                     [ 0.10,  0.10,  0.00],
                     [ 0.00, -0.20,  0.20]], device=device, requires_grad=True)  # (d_h, d_h)

b_h  = torch.tensor([[0.01],[0.02],[-0.01]], device=device, requires_grad=True)  # (d_h,1)

W_hy = torch.tensor([[ 0.30, -0.10,  0.20],
                     [-0.20,  0.10,  0.00]], device=device, requires_grad=True)  # (d_y, d_h)

b_y  = torch.tensor([[0.00],[0.05]], device=device, requires_grad=True)  # (d_y,1)

# Inputs X[t] shape (d_x,1)
X = [
    torch.tensor([[ 1.0],[ 0.0]], device=device),
    torch.tensor([[ 0.0],[ 1.0]], device=device),
    torch.tensor([[ 1.0],[ 1.0]], device=device),
]

# Targets as class indices for F.cross_entropy
targets = torch.tensor([0, 1, 1], device=device)  # length T

(W_xh.shape, W_hh.shape, b_h.shape, W_hy.shape, b_y.shape)

(torch.Size([3, 2]),
 torch.Size([3, 3]),
 torch.Size([3, 1]),
 torch.Size([2, 3]),
 torch.Size([2, 1]))

In [ ]:
# Forward pass (store intermediates)
h = torch.zeros((d_h, 1), device=device)

a_list, h_list, o_list = [], [], []
loss = 0.0

for t in range(T):
    a = W_xh @ X[t] + W_hh @ h + b_h     # (d_h,1)
    h = torch.tanh(a)                    # (d_h,1)
    o = W_hy @ h + b_y                   # (d_y,1)

    a_list.append(a); h_list.append(h); o_list.append(o)

    # cross_entropy expects (N,C) logits; we use N=1
    loss_t = F.cross_entropy(o.T, targets[t].view(1))
    loss = loss + loss_t

print("Total loss L =", float(loss))
for t in range(T):
    print(f"\n--- t={t+1} ---")
    print("a_t.T:", a_list[t].T)
    print("h_t.T:", h_list[t].T)
    print("o_t.T:", o_list[t].T)

Total loss L = 1.9313421249389648

--- t=1 ---
a_t.T: tensor([[ 0.1100,  0.0200, -0.1100]], grad_fn=<PermuteBackward0>)
h_t.T: tensor([[ 0.1096,  0.0200, -0.1096]], grad_fn=<PermuteBackward0>)
o_t.T: tensor([[0.0090, 0.0301]], grad_fn=<PermuteBackward0>)

--- t=2 ---
a_t.T: tensor([[-0.1571,  0.3330,  0.1641]], grad_fn=<PermuteBackward0>)
h_t.T: tensor([[-0.1559,  0.3212,  0.1626]], grad_fn=<PermuteBackward0>)
o_t.T: tensor([[-0.0463,  0.1133]], grad_fn=<PermuteBackward0>)

--- t=3 ---
a_t.T: tensor([[-0.1374,  0.3365,  0.0583]], grad_fn=<PermuteBackward0>)
h_t.T: tensor([[-0.1366,  0.3244,  0.0582]], grad_fn=<PermuteBackward0>)
o_t.T: tensor([[-0.0618,  0.1098]], grad_fn=<PermuteBackward0>)


/tmp/ipython-input-2124873022.py:18: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("Total loss L =", float(loss))


In [ ]:
# Backward (autograd)
loss.backward()

def show_grad(name, tensor):
    print(f"{name} grad:\n{tensor.grad}\n")

show_grad("W_xh", W_xh)
show_grad("W_hh", W_hh)
show_grad("b_h",  b_h)
show_grad("W_hy", W_hy)
show_grad("b_y",  b_y)

W_xh grad:
tensor([[ 0.0157,  0.4846],
        [-0.0085, -0.1880],
        [-0.0175,  0.1766]])

W_hh grad:
tensor([[-0.0064,  0.0773,  0.0080],
        [ 0.0011, -0.0284, -0.0017],
        [-0.0048,  0.0310,  0.0055]])

b_h grad:
tensor([[ 0.2760],
        [-0.1147],
        [ 0.0680]])

W_hy grad:
tensor([[-0.1895,  0.2860,  0.1568],
        [ 0.1895, -0.2860, -0.1568]])

b_y grad:
tensor([[ 0.4121],
        [-0.4121]])



## 7) Optional: finite-difference gradient check (one entry)

This numerically checks a single derivative to build trust in the math/autograd.

We approximate:
$$
\frac{\partial L}{\partial W_{hh}[0,0]} \approx \frac{L(W_{hh}[0,0]+\epsilon) - L(W_{hh}[0,0]-\epsilon)}{2\epsilon}
$$

In [ ]:
@torch.no_grad()
def forward_loss_given_params(W_xh_, W_hh_, b_h_, W_hy_, b_y_):
    h = torch.zeros((d_h, 1), device=device)
    L = 0.0
    for t in range(T):
        a = W_xh_ @ X[t] + W_hh_ @ h + b_h_
        h = torch.tanh(a)
        o = W_hy_ @ h + b_y_
        L = L + F.cross_entropy(o.T, targets[t].view(1))
    return float(L)

eps = 1e-4
i, j = 0, 0

W_hh_pos = W_hh.detach().clone()
W_hh_neg = W_hh.detach().clone()
W_hh_pos[i,j] += eps
W_hh_neg[i,j] -= eps

L_pos = forward_loss_given_params(W_xh.detach(), W_hh_pos, b_h.detach(), W_hy.detach(), b_y.detach())
L_neg = forward_loss_given_params(W_xh.detach(), W_hh_neg, b_h.detach(), W_hy.detach(), b_y.detach())

num_grad = (L_pos - L_neg) / (2*eps)
auto_grad = float(W_hh.grad[i,j])

print("Finite-diff approx:", num_grad)
print("Autograd value    :", auto_grad)
print("Abs error         :", abs(num_grad - auto_grad))

Finite-diff approx: -0.006556510925292969
Autograd value    : -0.0064475517719984055
Abs error         : 0.0001089591532945633
